In [1]:
import pandas as pd
import numpy as np
import wfdb
import ast

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

path = '/home/matrioszka/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return tuple(list(set(tmp)))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

In [9]:
y_train.unique()
# drop all rows with multiple labels
train_idx = [i for i, labels in enumerate(y_train) if len(labels) == 1]
X_train_single = X_train[train_idx]
y_train_single = y_train.iloc[train_idx].apply(lambda x: x[0])
# same for test set
test_idx = [i for i, labels in enumerate(y_test) if len(labels) == 1]
X_test_single = X_test[test_idx]
y_test_single = y_test.iloc[test_idx].apply(lambda x: x[0])

In [10]:
print(y_train.describe())
print(y_train_single.describe())
print(y_train_single.value_counts())
print(y_test.describe())
print(y_test_single.describe())
print(y_test_single.value_counts())

count       19634
unique         26
top       (NORM,)
freq         8170
Name: diagnostic_superclass, dtype: object
count     14620
unique        5
top        NORM
freq       8170
Name: diagnostic_superclass, dtype: object
diagnostic_superclass
NORM    8170
MI      2282
STTC    2163
CD      1525
HYP      480
Name: count, dtype: int64
count        2203
unique         23
top       (NORM,)
freq          913
Name: diagnostic_superclass, dtype: object
count     1652
unique       5
top       NORM
freq       913
Name: diagnostic_superclass, dtype: object
diagnostic_superclass
NORM    913
MI      256
STTC    243
CD      184
HYP      56
Name: count, dtype: int64


In [4]:
print(type(X_train))
print(np.shape(X))

<class 'numpy.ndarray'>
(21837, 1000, 12)


In [1]:
import matplotlib.pyplot as plt
plt.plot(X[10000, :102, 0])

NameError: name 'X' is not defined

In [ ]:
Y.describe()

In [12]:
np.save('X_train.npy', X_train_single)
np.save('y_train.npy', y_train_single)
np.save('X_test.npy', X_test_single)
np.save('y_test.npy', y_test_single)